# Can we predict next-month cancellation (`churn_next_month`)?

Plan (follows `checklist.md`):
1. Hold out the **latest months** as the test set, and keep **whole accounts** on one side (accounts present in the test period are removed from training).
2. Cross-validate inside training only: expanding-window time folds, with accounts in the validation block purged from the fold's training rows.
3. All preprocessing lives in a `Pipeline`, refit per fold.
4. Candidates (named before any scoring) and the selection rule are fixed below.
5. Choose model, hyperparameters and threshold on CV only; record the choice; then score the test set once.

**Features:** `lifetime_months` comes from a CRM export dated 2026-01-05, after every snapshot, so it is not known at prediction time (it encodes how long the account eventually lived, i.e. the outcome). It is dropped, as are `account_ref` (identifier) and `snapshot_month` (a calendar index cannot extrapolate to future months).

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
SEED=0
df=pd.read_csv('data.csv')
CAT=['plan']; NUM=['seats','monthly_fee','tenure_months','usage_hours','support_tickets']
FEATS=CAT+NUM; TARGET='churn_next_month'
print(df.shape, df.account_ref.nunique(), 'accounts; base rate', round(df[TARGET].mean(),3))
print(df.groupby('snapshot_month')[TARGET].mean().round(2).to_string())

The churn rate moves strongly over calendar time (peak around 2024-11 to 2025-01, ~8-11% recently), so a forward-in-time split is what matters, and the test base rate will differ from the training one.

## 1. Test set: latest months, whole accounts

In [ ]:
CUT='2025-08'
test_accts=set(df.loc[df.snapshot_month>=CUT,'account_ref'])
test=df[df.snapshot_month>=CUT].copy()
train=df[(df.snapshot_month<CUT)&~df.account_ref.isin(test_accts)].copy().reset_index(drop=True)
assert not set(train.account_ref)&set(test.account_ref)
assert train.snapshot_month.max()<test.snapshot_month.min()
print('train',len(train),'rate',round(train[TARGET].mean(),3),'| test',len(test),'rate',round(test[TARGET].mean(),3))
print('earlier rows dropped because the account is in the test set:',((df.snapshot_month<CUT)&df.account_ref.isin(test_accts)).sum())

## 2. Candidates, metric and selection rule (fixed before scoring)
- **baseline**: predicts the training prior (trivial).
- **logreg**: one-hot plan + standardised numerics, C in {0.01, 0.1, 1, 10}.
- **hgb**: gradient boosting, (max_depth, learning_rate) in {(2,0.05),(3,0.05),(3,0.1)}, 200 iterations, no early stopping.

Selection metric: mean PR-AUC across CV folds (churn is rare). The decision threshold is then chosen on the pooled out-of-fold predictions of the winner to maximise F1.

In [ ]:
def make_pre():
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),('n',StandardScaler(),NUM)])
def make_model(name,**p):
    if name=='baseline': m=DummyClassifier(strategy='prior')
    elif name=='logreg': m=LogisticRegression(C=p['C'],max_iter=1000)
    else: m=HistGradientBoostingClassifier(max_depth=p['d'],learning_rate=p['lr'],max_iter=200,early_stopping=False,random_state=SEED)
    return Pipeline([('pre',make_pre()),('m',m)])
CANDS=[('baseline',{})]+[('logreg',{'C':c}) for c in (0.01,0.1,1,10)]+[('hgb',{'d':d,'lr':l}) for d,l in ((2,.05),(3,.05),(3,.1))]
print(len(CANDS),'candidates')

## 3. Cross-validation inside training (expanding window, accounts purged)

In [ ]:
months=sorted(train.snapshot_month.unique())
# validation blocks: 4 consecutive 3-month blocks at the end of the training period
blocks=[months[i:i+3] for i in range(len(months)-12,len(months),3)]
def cv_folds():
    for b in blocks:
        va=train.snapshot_month.isin(b)
        va_accts=set(train.loc[va,'account_ref'])
        tr=(train.snapshot_month<b[0])&~train.account_ref.isin(va_accts)
        yield np.where(tr)[0],np.where(va)[0]
for tr,va in cv_folds(): print('fold train',len(tr),'val',len(va),train.snapshot_month.iloc[va].min(),'..',train.snapshot_month.iloc[va].max())

In [ ]:
X,y=train[FEATS],train[TARGET]
rows=[];oof={}
for name,p in CANDS:
    scores=[];pred=np.full(len(train),np.nan)
    for tr,va in cv_folds():
        m=make_model(name,**p).fit(X.iloc[tr],y.iloc[tr]); s=m.predict_proba(X.iloc[va])[:,1]
        pred[va]=s; scores.append(average_precision_score(y.iloc[va],s))
    oof[(name,str(p))]=pred
    rows.append(dict(model=name,params=str(p),cv_prauc=np.mean(scores),cv_prauc_sd=np.std(scores)))
cv=pd.DataFrame(rows).sort_values('cv_prauc',ascending=False)
cv

## 4. Choose model and threshold on CV only

In [ ]:
best=cv.iloc[0]; best_name=best.model; best_params=eval(best.params)
oof_pred=oof[(best_name,best.params)]; ok=~np.isnan(oof_pred)
grid=np.linspace(0.02,0.8,79)
oof_f1=[f1_score(y.values[ok],oof_pred[ok]>=t) for t in grid]; THRESHOLD=float(grid[int(np.argmax(oof_f1))])
print('chosen:',best_name,best_params,'| CV PR-AUC %.3f'%best.cv_prauc,'| threshold %.2f (OOF F1 %.3f)'%(THRESHOLD,max(oof_f1)))
print('baseline CV PR-AUC: %.3f'%cv[cv.model=='baseline'].cv_prauc.iloc[0])

**Decision recorded before the test set is touched:** the model, hyperparameters, features and threshold printed above are final.

## 5. Refit on all training rows; score the test set once

In [ ]:
final=make_model(best_name,**best_params).fit(X,y)
test_score=final.predict_proba(test[FEATS])[:,1]; y_test=test[TARGET]; test_label=(test_score>=THRESHOLD).astype(int)
base_score=make_model('baseline').fit(X,y).predict_proba(test[FEATS])[:,1]
headline=pd.Series(dict(roc_auc=roc_auc_score(y_test,test_score),pr_auc=average_precision_score(y_test,test_score),
  test_base_rate=y_test.mean(),precision=precision_score(y_test,test_label,zero_division=0),recall=recall_score(y_test,test_label),f1=f1_score(y_test,test_label),
  baseline_pr_auc=average_precision_score(y_test,base_score)))
headline.round(3)

## Conclusion
The table above is the headline (single test evaluation, compared with the trivial baseline whose PR-AUC equals the test base rate). Caveats: churn drifts strongly over time (training includes a high-churn winter, the test period is low-churn), so precision at the CV-chosen threshold may be miscalibrated; the test set is a single ~3k-row split over 4 months, so estimates are noisy; removing accounts that appear in the test period from training discards about a quarter of earlier rows.